In [2]:
import re
import sys, os
import numpy as np
from ccdproc import ImageFileCollection
from astropy.coordinates import SkyCoord
import astropy.units as u
from imageauxiliary_functions import get_filter_label
import json

In [ ]:
def table_expr_parser(column_expression, column_names, table_name='table'):

    #.Splitting expression using common operators
    split_expr = re.split(r'\s+[+\-<>=!&|]+\s+', column_expression)
    for side in split_expr:
        #..identifying variables in each expression side
        variables = set(re.findall(r'\d+\.\d+|\w+', side))
        #..replacing variables by column names
        for var in variables: 
            if var in column_names: 
                column_expression = column_expression.replace(var,table_name+"['"+var+"']")

    #.Enclosing both sides of boolean operators in parenthesis ()
    for sep in [' & ',' \| ']:
        side = re.split(sep, column_expression)
        if len(side) > 1:
            for j in range(len(side)): side[j] = '('+side[j]+')'
            sep = sep.replace('\\','')
            column_expression = sep.join(side)

    #.Return expression ready for evaluation
    return column_expression


def table_expr_keywords(column_expression):

    expression_keywords, possible_keywords = [], []
    
    for string_expr in column_expression:
        #.splitting conditions in expression by boolean operators
        joint_condition = re.split(r'[\s]*[&|][\s]*', string_expr)
        for expression in joint_condition:
            #..removing parenthesis and braces
            expression = re.sub(r'[()\[\]]','',expression)
            #..splitting left-hand-side (LHS) and right-hand-side (RHS) using common operators
            expression_side = re.split(r'\s+[<>=!?]+\s+', expression)
            for i,side in enumerate(expression_side):
                #...getting operands at the LHS
                if i == 0:
                    operands = re.split(r'\s+[\+\*-/]+\s+', side)     
                    expression_keywords.extend(operands)
                #...getting operands at the RHS
                else:
                    operands = re.findall(r'[a-zA-Z_][a-zA-Z0-9_\-\.]*',side)
                    if operands: possible_keywords.extend(operands)

    return list(set(expression_keywords)), list(set(possible_keywords))


def validate_header_keywords(header,keyword_list):

    valid = [keyw in header for keyw in keyword_list]

    for exception in ['offset','sequence']:
        if exception in keyword_list: 
            valid[keyword_list.index(exception)] = True

    return valid


class NpEncoder(json.JSONEncoder):
# Class to extend the JSONEncoder class

    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)
    

def create_image_sets(dataset, 
                      group_images_by=['object','filter1','filter2','sequence'],
                      separate_group_by = ['exptime <= 30 & offset < 0.02','exptime >= 100 & offset < 0.02'],
                      coordinate_keywords = ['ra','dec'],
                      imsets_file='imsets.json'):
    
    ifc_keywords = ['obstype','object','airmass','exptime']

    #.Processing input for demanded keywords
    group_keywords = group_images_by.copy()

    separate_keywords, possible_keywords = table_expr_keywords(separate_group_by)

    demanded_keywords = list(set(group_keywords+separate_keywords))
    if 'offset' in separate_keywords: demanded_keywords += coordinate_keywords

    mask = [re.match('filter.*',keyw) != None for keyw in demanded_keywords]
    filter_keywords = list(np.asarray(demanded_keywords)[mask])

    #----------------------------------------------------------------------------------
    #.If input dataset is a folder, expand it into a ImageFileCollection
    ifc_keywords = list(set(ifc_keywords))
    if isinstance(dataset,str): 
        ifc = ImageFileCollection(dataset, ext=0, keywords=ifc_keywords,
            glob_exclude="*master*.fits", glob_include="*.fits")    

    #.Validating demanded keywords in header
    header = next(ifc.headers())
    valid = validate_header_keywords(next(ifc.headers()),possible_keywords)
    for val,key in zip(valid,possible_keywords):
        if val and key not in demanded_keywords: demanded_keywords.append(key)

    valid = validate_header_keywords(next(ifc.headers()),demanded_keywords)
    if ~np.all(valid): 
        sys.exit(f'ABORTING: keywords not found: {list(np.array(demanded_keywords)[~np.array(valid)])}')
        
    ifc.keywords += demanded_keywords
    
    #.Generating main images table
    table = ifc.summary
    table['file'] = [os.path.join(ifc.location,file) for file in table['file']]

    #..adding sequence counter
    obj_list = np.array(table['object'])
    seq_number = 0
    seq_list = [seq_number]
    for i in np.arange(1,len(obj_list)):
        if obj_list[i] != obj_list[i-1]: seq_number += 1
        seq_list.append(seq_number)
        
    table['sequence'] = seq_list

    #----------------------------------------------------------------------------------
    #.Grouping exposures by keywords
    grouped = table.group_by(group_images_by)
    grouped_sizes = np.diff(grouped.groups.indices)

    #.Initiating image sets
    imsets = {}
    for i,group in enumerate(grouped.groups):

        #..calculating offsets for images in each group
        if grouped_sizes[i] > 1:
            coords = SkyCoord(ra= group[coordinate_keywords[0]], 
                              dec=group[coordinate_keywords[1]], 
                              unit=(u.hour,u.degree), frame='fk5')
            group['offset'] = np.round(coords[0].separation(coords).value, 7)

        #..separating groups by keyword expressions
        for separate in separate_group_by:

            #...evaluating expression
            eval_str = table_expr_parser(separate, demanded_keywords, table_name='group')
            mask = eval(eval_str)
            #...generating image set separated by this keyword expression
            if np.any(mask):
                imset_dict = {}
                for kwd in group_keywords:
                    imset_dict[kwd] = group[0][kwd]
                imset_dict['images'] = list(group['file'][mask])
                for kwd in separate_keywords:
                    imset_dict[kwd] = list(group[kwd][mask])

                imset_id = ''.join(group[0]['object'].split())
                imset_id+= '_'+get_filter_label(group[0][filter_keywords])
                seq = 1
                while imset_id+str(seq) in imsets: seq+=1
                imset_dict['output'] = str(os.path.join(ifc.location,imset_id+str(seq)+'.fits'))

                #....adding this imset to output dictionary
                imsets[imset_id+str(seq)] = imset_dict

    #----------------------------------------------------------------------------------
    #.Saving resulting image sets to output file (.json)
    with open(os.path.join(ifc.location,imsets_file),'w') as output_file:
        json.dump(imsets, output_file, indent=4, cls=NpEncoder)
        
    return imsets

In [7]:
dataset = './2021-10-29/' 

imsets = create_image_sets(dataset, group_images_by=['object','filter','filter2','sequence'])

imsets

{'ASS64_B-Bessel1': {'object': np.str_('ASS64'),
  'filter': np.str_('B-Bessel'),
  'filter2': np.str_('NO_FILTER'),
  'sequence': np.int64(6),
  'images': [np.str_('./2021-10-29/0275_SO2021B-018_1029.fits'),
   np.str_('./2021-10-29/0276_SO2021B-018_1029.fits')],
  'exptime': [np.float64(10.0), np.float64(10.0)],
  'offset': [np.float64(0.0), np.float64(7.16e-05)],
  'output': './2021-10-29/ASS64_B-Bessel1.fits'},
 'ASS64_B-Bessel2': {'object': np.str_('ASS64'),
  'filter': np.str_('B-Bessel'),
  'filter2': np.str_('NO_FILTER'),
  'sequence': np.int64(6),
  'images': [np.str_('./2021-10-29/0285_SO2021B-018_1029.fits'),
   np.str_('./2021-10-29/0286_SO2021B-018_1029.fits'),
   np.str_('./2021-10-29/0287_SO2021B-018_1029.fits')],
  'exptime': [np.float64(400.0), np.float64(400.0), np.float64(400.0)],
  'offset': [np.float64(4.11e-05), np.float64(4.85e-05), np.float64(2.22e-05)],
  'output': './2021-10-29/ASS64_B-Bessel2.fits'},
 'ASS64_H-alpha1': {'object': np.str_('ASS64'),
  'filter':

In [9]:
dataset = './2021-10-29/' 

group_images_by=['object','filter','filter2','sequence']
separate_group_by = ['exptime <= 30 & offset < 0.02','exptime >= 100 & offset < 0.02']
coordinate_keywords = ['ra','dec']
    
ifc_keywords = ['obstype','object','airmass','exptime']


In [10]:
#.Processing input for demanded keywords
group_keywords = group_images_by.copy()

separate_keywords, possible_keywords = table_expr_keywords(separate_group_by)

demanded_keywords = list(set(group_keywords+separate_keywords))
if 'offset' in separate_keywords: demanded_keywords += coordinate_keywords

mask = [re.match('filter.*',keyw) != None for keyw in demanded_keywords]
filter_keywords = list(np.asarray(demanded_keywords)[mask])

print(demanded_keywords)

['object', 'sequence', 'offset', 'filter', 'filter2', 'exptime', 'ra', 'dec']


In [11]:
#.If input dataset is a folder, expand it into a ImageFileCollection
ifc_keywords = list(set(ifc_keywords))
if isinstance(dataset,str): 
    ifc = ImageFileCollection(dataset, ext=0, keywords=ifc_keywords,
        glob_exclude="*master*.fits", glob_include="*.fits")    

#.Validating demanded keywords in header
valid = validate_header_keywords(next(ifc.headers()),possible_keywords)
for val,key in zip(valid,possible_keywords):
    if val and key not in demanded_keywords: demanded_keywords.append(key)

valid = validate_header_keywords(next(ifc.headers()),demanded_keywords)
if ~np.all(valid): 
    sys.exit(f'ABORTING: keywords not found: {list(np.array(demanded_keywords)[~np.array(valid)])}')
    
ifc.keywords += demanded_keywords
table = ifc.summary
table['file'] = [os.path.join(ifc.location,file) for file in table['file']]

#..adding sequence counter
obj_list = np.array(table['object'])
seq_number = 0
seq_list = [seq_number]
for i in np.arange(1,len(obj_list)):
    if obj_list[i] != obj_list[i-1]: seq_number += 1
    seq_list.append(seq_number)
    
table['sequence'] = seq_list

In [56]:
#.grouping exposures by keywords
grouped = table.group_by(group_images_by)
grouped_sizes = np.diff(grouped.groups.indices)

#.initiating image sets
imsets = {}
for i,group in enumerate(grouped.groups):

    #.calculating offsets for images in each group
    if grouped_sizes[i] > 1:
        coords = SkyCoord(ra= group[coordinate_keywords[0]], 
                          dec=group[coordinate_keywords[1]], 
                          unit=(u.hour,u.degree), frame='fk5')
        group['offset'] = np.round(coords[0].separation(coords).value, 7)

    #.separating groups by keyword expressions
    for separate in separate_group_by:

        #..evaluating expression
        eval_str = table_expr_parser(separate, demanded_keywords, table_name='group')
        mask = eval(eval_str)
        #..generating image set separated by expression
        if np.any(mask):
            imset_dict = {}
            for kwd in group_keywords:
                imset_dict[kwd] = group[0][kwd]
            imset_dict['images'] = list(group['file'][mask])
            for kwd in separate_keywords:
                imset_dict[kwd] = list(group[kwd][mask])

            imset_id = ''.join(group[0]['object'].split())
            imset_id+= '_'+get_filter_label(group[0][filter_keywords])
            seq = 1
            while imset_id+str(seq) in imsets: seq+=1
            imset_dict['output'] = imset_id+str(seq)+'.fits'

            #...adding this imset to output dictionary
            imsets[imset_id+str(seq)] = imset_dict



In [40]:
group['filter'].data.data

array(['V-Bessel', 'V-Bessel'], dtype='<U9')

In [38]:
table['exptime'].data.data[0]

np.float64(30.0)

In [57]:
imsets

{'ASS64_B-Bessel1': {'object': np.str_('ASS64'),
  'filter': np.str_('B-Bessel'),
  'filter2': np.str_('NO_FILTER'),
  'sequence': np.int64(6),
  'images': [np.str_('./2021-10-29/0275_SO2021B-018_1029.fits'),
   np.str_('./2021-10-29/0276_SO2021B-018_1029.fits')],
  'exptime': [np.float64(10.0), np.float64(10.0)],
  'offset': [np.float64(0.0), np.float64(7.16e-05)],
  'output': 'ASS64_B-Bessel1.fits'},
 'ASS64_B-Bessel2': {'object': np.str_('ASS64'),
  'filter': np.str_('B-Bessel'),
  'filter2': np.str_('NO_FILTER'),
  'sequence': np.int64(6),
  'images': [np.str_('./2021-10-29/0285_SO2021B-018_1029.fits'),
   np.str_('./2021-10-29/0286_SO2021B-018_1029.fits'),
   np.str_('./2021-10-29/0287_SO2021B-018_1029.fits')],
  'exptime': [np.float64(400.0), np.float64(400.0), np.float64(400.0)],
  'offset': [np.float64(4.11e-05), np.float64(4.85e-05), np.float64(2.22e-05)],
  'output': 'ASS64_B-Bessel2.fits'},
 'ASS64_H-alpha1': {'object': np.str_('ASS64'),
  'filter': np.str_('H-alpha'),
  'fi

In [9]:
with open(os.path.join(ifc.location,'imsets.json'),'w') as output_file:
    json.dump(imsets, output_file, indent=4, cls=NpEncoder)

In [10]:
with open(os.path.join(dataset,'imsets.json'),'r') as input_file:
    imset2=json.load(input_file)
imset2

{'ASS64_B-Bessel1': {'object': 'ASS64',
  'filter': 'B-Bessel',
  'filter2': 'NO_FILTER',
  'sequence': 6,
  'images': ['./2021-10-29/0275_SO2021B-018_1029.fits',
   './2021-10-29/0276_SO2021B-018_1029.fits'],
  'exptime': [10.0, 10.0],
  'offset': [0.0, 7.16e-05],
  'output': 'ASS64_B-Bessel1.fits'},
 'ASS64_B-Bessel2': {'object': 'ASS64',
  'filter': 'B-Bessel',
  'filter2': 'NO_FILTER',
  'sequence': 6,
  'images': ['./2021-10-29/0285_SO2021B-018_1029.fits',
   './2021-10-29/0286_SO2021B-018_1029.fits',
   './2021-10-29/0287_SO2021B-018_1029.fits'],
  'exptime': [400.0, 400.0, 400.0],
  'offset': [4.11e-05, 4.85e-05, 2.22e-05],
  'output': 'ASS64_B-Bessel2.fits'},
 'ASS64_H-alpha1': {'object': 'ASS64',
  'filter': 'H-alpha',
  'filter2': 'NO_FILTER',
  'sequence': 6,
  'images': ['./2021-10-29/0279_SO2021B-018_1029.fits',
   './2021-10-29/0280_SO2021B-018_1029.fits',
   './2021-10-29/0281_SO2021B-018_1029.fits'],
  'exptime': [120.0, 120.0, 120.0],
  'offset': [0.0, 5.93e-05, 0.0001